In [ ]:
# -*- coding: utf-8 -*-
"""cifar10lt_with_all_no_wine.ipynb"""

import os
import io
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from typing import Iterable
import math
import ssl
import itertools

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

RS_SHARED_TAU = 0.5

# ==========================================
# === Run config                          ==
# ==========================================
# Optional hyperparameter search uses one seed on the train/validation split.
# Final training reports mean and std over FINAL_SEEDS.
# Set SEARCH_EPOCHS lower for a quick smoke test if needed.
# Use RUN_HYPERPARAM_SEARCH=False to skip search and use DEFAULT_HPARAMS.
RUN_HYPERPARAM_SEARCH = True   # False => skip search and use DEFAULT_HPARAMS
HP_SEARCH_SEED = 42
FINAL_SEEDS = tuple(range(42, 52))
SEARCH_EPOCHS = 100
FINAL_EPOCHS = 100
HP_SEARCH_ALGOS = ("KL-RS",)  # use "all" to search every active algorithm
n_runs = len(FINAL_SEEDS)

# --- Subset selector: choose which algorithms / datasets to (re)train ---
# Set to None (or [] / "all") to run every algorithm in ALGOS. Otherwise
# list the subset you want. Names must match the entries in ALGOS below
# exactly (case-sensitive, including the unicode "χ²-DRO"). Useful when
# tweaking the hyperparameters of a single method — you can rerun just
# that method without retraining every baseline.
#
# Examples:
#   RUN_ALGOS = ["IRS"]
#   RUN_ALGOS = ["IRS", "KL-RS"]
#   RUN_ALGOS = None              # run everything
RUN_ALGOS    = None
RUN_DATASETS = None   # same convention; None / [] / "all" => every dataset


def download_content(url):
    context = ssl.create_default_context()
    context.check_hostname = False
    context.verify_mode = ssl.CERT_NONE
    req = urllib.request.Request(
        url,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    )
    return urllib.request.urlopen(req, context=context)

# =================================================================
# === KL-path helpers (shared by IRS and the corrected KL-RS)   ===
# =================================================================
# Both IRS (class-wise) and KL-RS (sample-wise) follow the same
# kappa-maximization template:
#
#     P(h)  ∝  P_hat * exp(h * F)            (KL-path away from P_hat)
#     κ(h)  =  (E_{P(h)}[F] - τ) / KL(P(h) || P_hat)
#     h*    =  argmax_h  κ(h)
#
# The only difference between IRS and KL-RS is that IRS works over
# the K class-buckets (F = class-wise mean loss, P_hat = empirical
# class prior), while KL-RS works over the B in-batch samples
# (F = per-sample loss, P_hat = uniform 1/B). The previous KL-RS
# implementation instead golden-sectioned the dual
#     g(λ) = λ * log E[exp((ℓ - τ)/λ)]
# which is monotonically decreasing in λ on (0, ∞) (no +λρ term,
# so no interior minimum), driving λ to lam_max and collapsing the
# objective to (mean(ℓ) - τ) — i.e., ERM with a constant offset.

@torch.no_grad()
def _kl_path_p_of_h(F_vals, P_hat, h, exp_clip=40.0):
    log_base = torch.log(P_hat.clamp_min(1e-30))
    logits = log_base + (F_vals * float(h))
    logits = torch.clamp(logits, min=logits.max()-exp_clip, max=logits.max()+1e-6)
    return torch.softmax(logits, dim=0)

@torch.no_grad()
def _secant_maximize_kappa(F_vals, P_hat, tau_t, h_left, h_right, max_iter=10):
    def K(h):
        P = _kl_path_p_of_h(F_vals, P_hat, h)
        E_F = torch.dot(P, F_vals)
        Dkl = (P * (torch.log(P.clamp_min(1e-30)) - torch.log(P_hat.clamp_min(1e-30)))).sum().item()
        return (E_F - tau_t).item() / max(1.0 * max(Dkl, 0.0) + 1e-2, 1e-20), P

    a, b = float(h_left), float(h_right)
    Ka, _ = K(a); Kb, _ = K(b)
    for _ in range(max_iter):
        if abs(b - a) < 1e-8: break
        m = (Kb - Ka) / (b - a)
        c = b - Kb / m if abs(m) > 1e-12 else b - np.sign(Kb)*0.1
        Kc, P_star = K(c)
        if Kc > Kb: a, Ka = b, Kb; b, Kb = c, Kc
        else: a, Ka = c, Kc
    return b, P_star

def classwise_F(loss_vec, g, n_groups, P_glob):
    dev = loss_vec.device
    counts = torch.bincount(g, minlength=n_groups).to(dev)
    mask = counts > 0
    lsum = torch.zeros(n_groups, device=dev).scatter_add_(0, g, loss_vec)
    F = torch.zeros(n_groups, device=dev); F[mask] = lsum[mask] / counts[mask].float()
    P_hatS = torch.zeros_like(F); Pg = P_glob.to(dev)
    P_hatS[mask] = Pg[mask] / Pg[mask].sum().clamp_min(1e-12)
    return F, P_hatS


# =================================================================
# === KL-RS helpers                                            ===
# =================================================================
# KL-RS (the real algorithm, Algorithm 1 + 2 from the paper) is the
# *class-level* method. It minimizes the KL-feasibility objective
#     L(theta; lam, tau) = sum_g q_g * exp((l_g - tau) / lam)
# where q_g is the empirical class prior and l_g is the per-class
# mean loss. The smallest lam at which the trained model satisfies
# L <= 1 is found by an outer doubling-then-bisection loop, with
# inner_epochs of retraining per candidate lam.
#
# This is structurally different from IRS, which (in its current
# instance-wise form) does a per-batch kappa-maximization at sample
# granularity with no outer retraining loop.

def _klrs_feasibility_loss(loss_vec, g, P_g, lam, tau, n_groups=5):
    """Per-batch KL-RS training loss at fixed (lam, tau).

    Returns sum_g q_g * exp((l_g - tau) / lam) computed over the
    classes present in the batch (q_g is P_g renormalized over
    present classes). Falls back to mean loss when fewer than 2
    classes are present.
    """
    F, P = classwise_F(loss_vec, g, n_groups, P_g)
    present_mask = P > 0
    if present_mask.sum() < 2:
        return loss_vec.mean()
    P_pres = P[present_mask]
    F_pres = F[present_mask]
    P_renorm = P_pres / P_pres.sum().clamp_min(1e-12)
    exponents = ((F_pres - tau) / max(float(lam), 1e-8)).clamp(max=50.0)
    return torch.dot(P_renorm.detach(), torch.exp(exponents))

@torch.no_grad()
def _check_klrs_feasibility(model, loader, P_g, lam, tau, n_groups,
                             device, crit):
    """Full-dataset feasibility test: returns (is_feasible, feas_val).

    feas_val = sum_g q_g * exp((l_g - tau) / lam) evaluated on `loader`.
    Feasible <=> feas_val <= 1.
    """
    was_training = model.training
    model.eval()
    loss_sum = torch.zeros(n_groups, device=device)
    count    = torch.zeros(n_groups, device=device)
    for x, y, _, g in loader:
        x, y, g = x.to(device), y.to(device), g.to(device)
        lv = crit(model(x), y).squeeze(-1)
        loss_sum.scatter_add_(0, g, lv)
        count   .scatter_add_(0, g, torch.ones_like(g, dtype=torch.float))
    present  = count > 0
    group_l  = loss_sum[present] / count[present].clamp_min(1.0)
    prior    = P_g[present]
    prior    = prior / prior.sum().clamp_min(1e-12)
    exponents = ((group_l - tau) / max(float(lam), 1e-8)).clamp(max=50.0)
    feas_val = float(torch.dot(prior, torch.exp(exponents)).item())
    if was_training:
        model.train()
    return feas_val <= 1.0, feas_val

def _train_klrs_protocol(model, train_loader, opt, crit, P_g, device,
                          epochs, n_groups=5, alt_iters=5):
    """Real KL-RS: ERM warmup -> doubling -> bisection -> finalize.

    Budget allocation (chosen to match the other algorithms' total of
    `epochs` epochs):
        warmup_epochs    = 10
        alt_iters        = number of outer lambda checks
        inner_epochs     = floor(remaining budget / alt_iters)
        Any leftover budget is spent at the final lambda.

    Any epochs left unused after bisection converges (interval < tol)
    are spent training at the final lambda* so the full budget is used.
    """
    # ---- Budget ----
    warmup_epochs    = max(1, epochs // 10)               # 10 of 100
    max_bisect_steps = max(1, int(alt_iters))
    remaining_epochs = max(0, epochs - warmup_epochs)
    inner_epochs     = max(1, remaining_epochs // max_bisect_steps)
    # ---- Algorithm constants (match klrs_tabular_new.ipynb) ----
    target_tau        = RS_SHARED_TAU
    lambda_init       = 1.0
    lambda_bisect_tol = 0.01

    def _train_one_epoch(lam):
        model.train()
        for x, y, _, g in train_loader:
            x, y, g = x.to(device), y.to(device), g.to(device)
            loss_vec = crit(model(x), y).squeeze(-1)
            if lam is None:  # ERM warmup
                loss = loss_vec.mean()
            else:
                loss = _klrs_feasibility_loss(loss_vec, g, P_g,
                                              lam, target_tau, n_groups)
            opt.zero_grad(); loss.backward(); opt.step()

    # ---- Phase 1: ERM warmup ----
    for _ in range(warmup_epochs):
        _train_one_epoch(lam=None)
    epochs_used = warmup_epochs

    # ---- Phase 2a: Doubling — find a feasible upper bound ----
    lam_lower, lam_upper = 0.0, lambda_init
    bisect_steps = 0
    while bisect_steps < max_bisect_steps and epochs_used < epochs:
        bisect_steps += 1
        steps_this_round = min(inner_epochs, epochs - epochs_used)
        for _ in range(steps_this_round):
            _train_one_epoch(lam=lam_upper)
        epochs_used += steps_this_round
        is_feas, _ = _check_klrs_feasibility(
            model, train_loader, P_g, lam_upper, target_tau,
            n_groups, device, crit)
        if is_feas:
            break
        lam_lower  = lam_upper
        lam_upper *= 2.0

    # ---- Phase 2b: Bisection — narrow [lam_lower, lam_upper] ----
    while ((lam_upper - lam_lower) >= lambda_bisect_tol
           and bisect_steps < max_bisect_steps
           and epochs_used < epochs):
        lam_mid = 0.5 * (lam_upper + lam_lower)
        bisect_steps += 1
        steps_this_round = min(inner_epochs, epochs - epochs_used)
        for _ in range(steps_this_round):
            _train_one_epoch(lam=lam_mid)
        epochs_used += steps_this_round
        is_feas, _ = _check_klrs_feasibility(
            model, train_loader, P_g, lam_mid, target_tau,
            n_groups, device, crit)
        if is_feas:
            lam_upper = lam_mid
        else:
            lam_lower = lam_mid

    # ---- Phase 3: Spend any remaining budget at lambda* ----
    # Keeps total epochs == `epochs` even if bisection converged early.
    lam_star = 0.5 * (lam_upper + lam_lower)
    while epochs_used < epochs:
        _train_one_epoch(lam=lam_star)
        epochs_used += 1

    return model

# ==========================================
# === SAM Optimizer                      ===
# ==========================================
class SAMSGD(torch.optim.SGD):
    """Sharpness-Aware Minimization on top of SGD-with-momentum.

    Matches the CIFAR experiment (standard Foret et al. 2021 SAM uses
    SGD as the base optimizer). The previous version inherited from Adam.
    NOTE: hyperparameters in Table 12 were selected against Adam-SAM —
    after this swap the optimal `lr` may shift (SGD typically needs a
    larger lr than Adam at the same problem)."""

    def __init__(self, params, lr=1e-3, rho=0.05, momentum=0.9, **kwargs):
        super().__init__(params, lr=lr, momentum=momentum, **kwargs)
        self.rho = rho

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()
        super().step()
        return loss

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        scale = self.rho / (grad_norm + 1e-12)
        self.e_w = []
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: self.e_w.append(None); continue
                e_w = p.grad * scale.to(p)
                p.add_(e_w); self.e_w.append(e_w)
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        idx = 0
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: idx+=1; continue
                p.sub_(self.e_w[idx]); idx+=1
        super().step()
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def _grad_norm(self):
        norms = [p.grad.norm(p=2) for group in self.param_groups for p in group["params"] if p.grad is not None]
        return torch.norm(torch.stack(norms), p=2) if norms else torch.tensor(0.)

# ==========================================
# === Data Loaders                       ===
# ==========================================
class RobustDataset(Dataset):
    def __init__(self, x, y, g):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)
        self.g = torch.tensor(g, dtype=torch.long)
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i], torch.tensor(i).long(), self.g[i]

def disc_g(y, bins=5): return pd.qcut(y.flatten(), q=bins, labels=False, duplicates='drop')

def split_train_val(x, y, g, val_ratio=0.2):
    n = len(x)
    p = np.random.permutation(n)
    n_val = int(n * val_ratio)
    idx_val = p[:n_val]; idx_train = p[n_val:]
    return RobustDataset(x[idx_train], y[idx_train], g[idx_train]), \
           RobustDataset(x[idx_val], y[idx_val], g[idx_val]), \
           RobustDataset(x, y, g)

def get_bike():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip"
    try:
        with zipfile.ZipFile(io.BytesIO(download_content(url).read())) as z:
            with z.open("hour.csv") as f: df = pd.read_csv(f)
    except Exception as e: print(f"Bike Error: {e}"); return None
    X = StandardScaler().fit_transform(df[['season','mnth','hr','holiday','weekday','workingday','weathersit','temp','atemp','hum','windspeed']].values)
    y = (df['cnt'].values.astype(float) - df['cnt'].mean()) / df['cnt'].std()
    idx1 = np.where((df['yr']==0) & (df['season'].isin([1,2])))[0]
    idx2 = np.where((df['yr']==0) & (df['season'].isin([3,4])))[0]
    idxt = np.where(df['yr']==1)[0]
    t1, v1, f1 = split_train_val(X[idx1], y[idx1], disc_g(y[idx1]))
    t2, v2, f2 = split_train_val(X[idx2], y[idx2], disc_g(y[idx2]))
    dt = RobustDataset(X[idxt], y[idxt], disc_g(y[idxt]))
    return (t1, v1, f1), (t2, v2, f2), dt

def get_concrete():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"
    try: df = pd.read_excel(io.BytesIO(download_content(url).read()))
    except Exception as e: print(f"Concrete Error: {e}"); return None
    y = (df.iloc[:, -1].values - df.iloc[:, -1].mean()) / df.iloc[:, -1].std()
    age = df.iloc[:, 7].values
    X = StandardScaler().fit_transform(df.iloc[:, :-2].values)
    idx1 = np.where(age<28)[0]; idx2 = np.where(age==28)[0]; idxt = np.where(age>28)[0]
    if len(idx1)<10: p=np.random.permutation(len(y)); idx1,idx2,idxt = p[:int(0.4*len(y))], p[int(0.4*len(y)):int(0.8*len(y))], p[int(0.8*len(y)):]
    t1, v1, f1 = split_train_val(X[idx1], y[idx1], disc_g(y[idx1]))
    t2, v2, f2 = split_train_val(X[idx2], y[idx2], disc_g(y[idx2]))
    dt = RobustDataset(X[idxt], y[idxt], disc_g(y[idxt]))
    return (t1, v1, f1), (t2, v2, f2), dt

def get_loader(name):
    loaders = {"bike":get_bike, "concrete":get_concrete}
    if name not in loaders: return None
    res = loaders[name]()
    if res is None: return None
    return res

# ==========================================
# === Models                             ===
# ==========================================
class MLP(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        )
    def forward(self, x): return self.net(x)

class Poly(nn.Module):
    def __init__(self, d_in, degree=2):
        super().__init__()
        self.l = nn.Linear(d_in*degree, 1); self.d = degree
    def forward(self, x): return self.l(torch.cat([x.pow(i) for i in range(1,self.d+1)],1))

def get_model(name, d_in):
    if name == "bike": return Poly(d_in)
    return MLP(d_in)

# ==========================================
# === DRO Loss Functions                 ===
# ==========================================
def chi2_dro_loss(loss_vec: torch.Tensor, rho: float,
                  bisect_tol: float = 1e-5, max_iter: int = 50) -> torch.Tensor:
    B = loss_vec.size(0)
    if B == 0:
        return loss_vec.new_tensor(0.0)

    rho = max(float(rho), 0.0)
    c = math.sqrt(1.0 + 2.0 * rho)

    eta_min = loss_vec.min().item()
    eta_max = loss_vec.max().item()

    if abs(eta_max - eta_min) < 1e-12:
        return loss_vec.mean()

    def eval_R(eta):
        diff = loss_vec - eta
        positive_part = torch.clamp(diff, min=0.0)
        var_term = (positive_part ** 2).mean()
        return c * torch.sqrt(var_term + 1e-12) + eta

    for _ in range(max_iter):
        if (eta_max - eta_min) < bisect_tol:
            break
        eta_mid = 0.5 * (eta_min + eta_max)
        diff = loss_vec - eta_mid
        positive_part = torch.clamp(diff, min=0.0)
        var_term = (positive_part ** 2).mean()
        grad = 1.0 - c * (positive_part.mean()) / torch.sqrt(var_term + 1e-12)
        if grad.item() < 0:
            eta_min = eta_mid
        else:
            eta_max = eta_mid

    eta_star = 0.5 * (eta_min + eta_max)
    return eval_R(eta_star)


def cvar_loss_from_batch(loss_vec: torch.Tensor, alpha: float) -> torch.Tensor:
    alpha = max(min(float(alpha), 1.0), 1e-6)
    B = loss_vec.size(0)
    k = max(1, int(math.ceil(alpha * B)))
    topk_losses, _ = torch.topk(loss_vec, k, largest=True, sorted=False)
    return topk_losses.mean()


class GroupDROLossComputer:
    def __init__(self, n_groups=5, alpha=0.2, gamma=0.1, device='cuda'):
        self.alpha = alpha
        self.gamma = gamma
        self.device = device
        self.n_groups = n_groups

        self.adv_probs = torch.ones(self.n_groups, device=device) / self.n_groups
        self.exp_avg_loss = torch.zeros(self.n_groups, device=device)
        self.exp_avg_initialized = torch.zeros(self.n_groups, device=device).bool()

    def loss(self, loss_vec, group_idx, is_training=True):
        group_losses = []
        group_counts_batch = []
        for g in range(self.n_groups):
            mask = (group_idx == g)
            if mask.any():
                group_loss = loss_vec[mask].mean()
                group_losses.append(group_loss)
                group_counts_batch.append(mask.sum().item())
            else:
                group_losses.append(torch.tensor(0.0, device=self.device))
                group_counts_batch.append(0)

        group_losses = torch.stack(group_losses)
        group_counts_batch = torch.tensor(group_counts_batch, dtype=torch.float32, device=self.device)

        if is_training:
            for g in range(self.n_groups):
                if group_counts_batch[g] > 0:
                    if not self.exp_avg_initialized[g]:
                        self.exp_avg_loss[g] = group_losses[g].detach()
                        self.exp_avg_initialized[g] = True
                    else:
                        self.exp_avg_loss[g] = (
                            self.gamma * group_losses[g].detach() +
                            (1 - self.gamma) * self.exp_avg_loss[g]
                        )

            adv_probs = torch.exp(self.alpha * self.exp_avg_loss.detach())
            adv_probs = adv_probs / (adv_probs.sum() + 1e-12)
            self.adv_probs = adv_probs

        weighted_loss = (self.adv_probs * group_losses).sum()
        return weighted_loss

# ==========================================
# === Train Loop                         ===
# ==========================================
def train_loop(model, train_loader, env_loaders, algo, param_dict, epochs=100):
    lr = param_dict['lr']
    penalty = param_dict.get('penalty', 0.0)
    rho_sam = param_dict.get('rho', 0.05)
    lam_rex = param_dict.get('lam', 1.5)
    klrs_alt_iters = param_dict.get('alt_iters', 5)
    chi2_rho = param_dict.get('chi2_rho', 0.1)
    cvar_alpha = param_dict.get('cvar_alpha', 0.1)
    groupdro_alpha = param_dict.get('groupdro_alpha', 0.2)
    rex_beta = param_dict.get('rex_beta', 1.0)

    target_tau_factor = 1.01
    tau_frozen = False
    current_tau = None

    if algo == "SAM":
        opt = SAMSGD(model.parameters(), lr=lr, rho=rho_sam, momentum=0.9, weight_decay=0.0)
    elif algo == "ERM-SGD":
        opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=0.0, momentum=0.9)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.0)

    crit = nn.MSELoss(reduction='none')

    P_g = torch.zeros(5).to(device)
    for _,_,_,g in train_loader:
        g = g.to(device)
        P_g.scatter_add_(0, g, torch.ones_like(g, dtype=torch.float, device=device))
    P_g /= P_g.sum()

    # KL-RS uses an outer doubling+bisection protocol over lambda, with
    # multiple retraining rounds at each lambda candidate. Total epoch
    # budget = epochs (same as every other algorithm). Hand off to the
    # dedicated routine here — the per-batch elif chain below is only
    # for online algorithms.
    if algo == "KL-RS":
        return _train_klrs_protocol(
            model, train_loader, opt, crit, P_g, device, epochs,
            n_groups=5, alt_iters=klrs_alt_iters,
        )


    if algo == "GroupDRO":
        groupdro_computer = GroupDROLossComputer(n_groups=5, alpha=groupdro_alpha, device=device)

    l1, l2 = env_loaders

    for ep in range(epochs):
        model.train()

        if algo in ["V-REx", "MM-REx", "IRM"]:
            for (x1,y1,_,_), (x2,y2,_,_) in zip(l1, l2):
                x1,y1,x2,y2 = x1.to(device), y1.to(device), x2.to(device), y2.to(device)
                l1_v, l2_v = crit(model(x1), y1).mean(), crit(model(x2), y2).mean()

                if algo == "V-REx":
                    penalty_weight = min(1.0, float(ep) / 10.0) if ep <= 10 else 1.0
                    var_penalty = (l1_v - l2_v)**2
                    loss = (l1_v+l2_v)/2 + penalty_weight * rex_beta * var_penalty
                elif algo == "MM-REx":
                    # ------------------------------------------------------------
                    # MM-REx (Krueger et al., 2021):
                    #   R_MM-REx(λ_mm) = λ_mm * max_e R_e + (1 - λ_mm) * mean_e R_e
                    # with λ_mm ≥ 0, and λ_mm > 1 being the "extrapolation" regime
                    # (more aggressive than vanilla min–max, which is λ_mm = 1).
                    #
                    # The previous code used `(1 - 2c)*max + c*sum`, which is the
                    # right algebra ONLY if c = (1 - λ_mm)/2. The grid fed in
                    # `c ∈ {1.0, 1.5, 2.0, 3.0}` directly, so at c=1 the loss
                    # became sum − max = min(R_1, R_2): minimizing it *raised*
                    # the smaller env's loss — exactly backwards.
                    # ------------------------------------------------------------
                    penalty_weight = min(1.0, float(ep) / 10.0) if ep <= 10 else 1.0
                    # Warm from ERM (λ_mm = 0 ⇒ pure mean) up to lam_rex.
                    current_lambda = penalty_weight * lam_rex
                    l_val = torch.stack([l1_v, l2_v])
                    loss = current_lambda * l_val.max() + (1.0 - current_lambda) * l_val.mean()
                elif algo == "IRM":
                    penalty_weight = min(1.0, float(ep) / 10.0) if ep <= 10 else 1.0
                    dum = torch.tensor(1., requires_grad=True, device=device)
                    g1 = torch.autograd.grad(crit(model(x1)*dum, y1).mean(), dum, create_graph=True)[0]
                    g2 = torch.autograd.grad(crit(model(x2)*dum, y2).mean(), dum, create_graph=True)[0]
                    loss = (l1_v+l2_v)/2 + penalty_weight * penalty * (g1**2 + g2**2)
                opt.zero_grad(); loss.backward(); opt.step()
        else:
            for x,y,_,g in train_loader:
                x,y,g = x.to(device), y.to(device), g.to(device)
                loss_vec = crit(model(x), y).squeeze()

                if algo in ["ERM", "ERM-SGD"]:
                    loss = loss_vec.mean()
                elif algo == "SAM":
                    loss = loss_vec.mean(); loss.backward()
                    opt.first_step(zero_grad=True); crit(model(x), y).mean().backward(); opt.second_step(zero_grad=True); continue

                elif algo == "IRS":
                    # ------------------------------------------------------------
                    # IRS (Iterative Robust Satisficing) — instance-wise variant.
                    #
                    # Per-batch kappa-maximization over the B samples in the
                    # batch (atoms = individual samples, base distribution =
                    # uniform 1/B). The KL-path P(h)_i ~ (1/B) * exp(h * l_i)
                    # has B degrees of freedom, so the adversary can isolate
                    # the genuinely hard examples rather than averaging within
                    # pre-defined class buckets.
                    #
                    #   P(h)  prop  (1/B) * exp(h * l_i)
                    #   kappa(h)  =  (E_{P(h)}[l] - tau) / KL(P(h) || uniform)
                    # ------------------------------------------------------------
                    losses_det = loss_vec.detach()
                    B = losses_det.size(0)
                    P_hat = torch.full((B,), 1.0 / B,
                                       device=losses_det.device,
                                       dtype=losses_det.dtype)
                    if tau_frozen:
                        target = current_tau
                    else:
                        adaptive_tau = losses_det.mean().item() * target_tau_factor
                        if adaptive_tau <= RS_SHARED_TAU:
                            tau_frozen = True
                            current_tau = RS_SHARED_TAU
                            target = RS_SHARED_TAU
                        else:
                            target = adaptive_tau
                    _, P_star = _secant_maximize_kappa(
                        losses_det, P_hat,
                        losses_det.new_tensor(target), -2, 2
                    )
                    loss = (loss_vec * (P_star / (P_hat + 1e-12)).detach()).mean()

                elif algo == "GroupDRO":
                    loss = groupdro_computer.loss(loss_vec, g, is_training=True)

                elif algo == "χ²-DRO":
                    loss = chi2_dro_loss(loss_vec, chi2_rho)

                elif algo == "CVaR-DRO":
                    loss = cvar_loss_from_batch(loss_vec, cvar_alpha)

                opt.zero_grad(); loss.backward(); opt.step()
    return model

def evaluate(model, loader):
    model.eval()
    loss_sum = 0; tot = 0
    with torch.no_grad():
        for x,y,_,_ in loader:
            x,y = x.to(device), y.to(device)
            loss_sum += nn.MSELoss(reduction='sum')(model(x), y).item()
            tot += x.size(0)
    return loss_sum / tot

# ==========================================
# === Default hyperparameters            ===
# ==========================================
# Edit this block to change the defaults used when
# RUN_HYPERPARAM_SEARCH=False. When search is enabled, these values are
# also included as candidates so the search cannot accidentally omit the
# current camera-ready setting.
#
# Camera-ready defaults: train each
# (dataset, algo) once on the full data (train + val combined, i.e. the
# full_* loaders) with the hyperparameters below — taken from Table 12.
#
# Note on KL-RS: Table 12 lists `alt_iters=2`, but the corrected sample-
# level KL-RS (mirroring IRS via _secant_maximize_kappa) has no alt_iters
# knob — only `lr` is tunable. The `alt_iters` column in Table 12 should
# be removed for KL-RS to stay consistent with the implementation.

DATASETS = ["bike", "concrete"]
ALGOS = ["ERM", "ERM-SGD", "SAM", "IRS", "KL-RS",
         "V-REx", "MM-REx", "IRM", "GroupDRO", "χ²-DRO", "CVaR-DRO"]

DEFAULT_HPARAMS = {
    "bike": {
        "ERM":      {"lr": 1e-3},
        "ERM-SGD":  {"lr": 1e-3},
        "SAM":      {"lr": 1e-3, "rho": 0.05},
        "IRS":      {"lr": 1e-2},
        "KL-RS":    {"lr": 1e-2, "alt_iters": 5},
        "V-REx":    {"lr": 1e-3, "rex_beta": 0.1},
        "MM-REx":   {"lr": 1e-2, "lam": 1.0},
        "IRM":      {"lr": 1e-3, "penalty": 0.01},
        "GroupDRO": {"lr": 1e-3, "groupdro_alpha": 0.1},
        "χ²-DRO":   {"lr": 1e-2, "chi2_rho": 0.5},
        "CVaR-DRO": {"lr": 1e-2, "cvar_alpha": 0.5},
    },
    "concrete": {
        "ERM":      {"lr": 1e-4},
        "ERM-SGD":  {"lr": 1e-3},
        "SAM":      {"lr": 1e-4, "rho": 0.2},
        "IRS":      {"lr": 1e-4},
        "KL-RS":    {"lr": 5e-3, "alt_iters": 5},
        "V-REx":    {"lr": 1e-4, "rex_beta": 0.1},
        "MM-REx":   {"lr": 1e-4, "lam": 1.0},
        "IRM":      {"lr": 1e-4, "penalty": 0.1},
        "GroupDRO": {"lr": 1e-4, "groupdro_alpha": 0.5},
        "χ²-DRO":   {"lr": 1e-4, "chi2_rho": 0.1},
        "CVaR-DRO": {"lr": 1e-4, "cvar_alpha": 0.3},
    },
}

# Hyperparameter grids. Edit these to change the validation search space.
# DEFAULT_HPARAMS values are automatically prepended to each grid.
BASE_LR_GRID = [1e-4, 1e-3, 1e-2]
HYPERPARAM_GRIDS = {
    d: {algo: {"lr": BASE_LR_GRID} for algo in ALGOS}
    for d in DATASETS
}
for d in DATASETS:
    HYPERPARAM_GRIDS[d]["KL-RS"].update({"alt_iters": [1, 2, 3, 5, 10, 15]})
    HYPERPARAM_GRIDS[d]["SAM"].update({"rho": [0.05, 0.1, 0.2]})
    HYPERPARAM_GRIDS[d]["V-REx"].update({"rex_beta": [0.1, 1.0, 10.0]})
    HYPERPARAM_GRIDS[d]["MM-REx"].update({"lam": [0.5, 1.0, 1.5, 2.0]})
    HYPERPARAM_GRIDS[d]["IRM"].update({"penalty": [0.01, 0.1, 1.0, 10.0]})
    HYPERPARAM_GRIDS[d]["GroupDRO"].update({"groupdro_alpha": [0.1, 0.2, 0.5]})
    HYPERPARAM_GRIDS[d][ALGOS[-2]].update({"chi2_rho": [0.1, 0.5, 1.0]})
    HYPERPARAM_GRIDS[d]["CVaR-DRO"].update({"cvar_alpha": [0.1, 0.3, 0.5]})

# ==========================================
# === Run Experiments                    ===
# ==========================================
def _resolve_subset(selected, full_list, label):
    if selected is None or selected == [] or selected == "all":
        return list(full_list)
    if isinstance(selected, str):
        selected = [selected]
    unknown = [a for a in selected if a not in full_list]
    if unknown:
        raise ValueError(
            f"Unknown {label} in selector: {unknown}. "
            f"Allowed values: {full_list}")
    # preserve the user-provided order (so you can prioritize the algo
    # you're actively tuning at the top of the printout)
    return list(selected)

def set_all_seeds(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def _as_list(value):
    if isinstance(value, (list, tuple)):
        return list(value)
    return [value]

def _candidate_params(default_params, grid):
    if not grid:
        return [dict(default_params)]
    keys = list(grid.keys())
    value_lists = []
    for key in keys:
        vals = _as_list(grid[key])
        if key in default_params and all(v != default_params[key] for v in vals):
            vals = [default_params[key]] + vals
        value_lists.append(vals)
    candidates = []
    seen = set()
    for combo in itertools.product(*value_lists):
        params = dict(default_params)
        params.update({k: v for k, v in zip(keys, combo)})
        sig = tuple(sorted(params.items()))
        if sig not in seen:
            seen.add(sig)
            candidates.append(params)
    return candidates

def _run_hyperparameter_search(dname, d_in, algo, t1, t2, v1, v2):
    default_params = DEFAULT_HPARAMS[dname][algo]
    grid = HYPERPARAM_GRIDS.get(dname, {}).get(algo, {})
    candidates = _candidate_params(default_params, grid)
    train_l1 = DataLoader(t1, 256, True)
    train_l2 = DataLoader(t2, 256, True)
    train_all = DataLoader(torch.utils.data.ConcatDataset([t1, t2]), 256, True)
    val_loader = DataLoader(torch.utils.data.ConcatDataset([v1, v2]), 256, False)

    best_params = None
    best_score = float("inf")
    search_rows = []
    print(f"  [{algo}] hyperparameter search: {len(candidates)} candidates")
    for cand_idx, params in enumerate(candidates, start=1):
        set_all_seeds(HP_SEARCH_SEED)
        m_search = get_model(dname, d_in).to(device)
        m_search = train_loop(
            m_search, train_all, (train_l1, train_l2),
            algo, params, epochs=SEARCH_EPOCHS,
        )
        val_mse = evaluate(m_search, val_loader)
        search_rows.append({
            "dataset": dname,
            "algo": algo,
            "candidate": cand_idx,
            "val_mse": val_mse,
            "params": dict(params),
        })
        print(f"    cand {cand_idx:02d}/{len(candidates):02d}: val={val_mse:.4f}, params={params}")
        if val_mse < best_score:
            best_score = float(val_mse)
            best_params = dict(params)
        del m_search
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print(f"  [{algo}] selected params={best_params} (val={best_score:.4f})")
    return best_params, best_score, search_rows

ACTIVE_ALGOS    = _resolve_subset(RUN_ALGOS,    ALGOS,    "algo")
ACTIVE_DATASETS = _resolve_subset(RUN_DATASETS, DATASETS, "dataset")
ACTIVE_HP_SEARCH_ALGOS = _resolve_subset(HP_SEARCH_ALGOS, ALGOS, "hyperparameter-search algo")
ACTIVE_HP_SEARCH_ALGOS = [a for a in ACTIVE_HP_SEARCH_ALGOS if a in ACTIVE_ALGOS]

SEARCH_MODE = "validation search" if RUN_HYPERPARAM_SEARCH else "default hyperparameters"
print(f"{'='*60}")
print(f"RUNNING TABULAR EXPERIMENTS ({SEARCH_MODE}, n_runs={n_runs})")
print(f"Datasets : {ACTIVE_DATASETS}")
print(f"Algorithms: {ACTIVE_ALGOS}")
print(f"Hyperparameter search algorithms: {ACTIVE_HP_SEARCH_ALGOS if RUN_HYPERPARAM_SEARCH else []}")
print(f"Final seeds: {FINAL_SEEDS}")
print(f"Search seed: {HP_SEARCH_SEED}, search epochs: {SEARCH_EPOCHS}, final epochs: {FINAL_EPOCHS}")
if len(ACTIVE_ALGOS) < len(ALGOS) or len(ACTIVE_DATASETS) < len(DATASETS):
    print(f"  (subset mode — skipping "
          f"{[a for a in ALGOS if a not in ACTIVE_ALGOS]} algos, "
          f"{[d for d in DATASETS if d not in ACTIVE_DATASETS]} datasets)")
print(f"{'='*60}")
RESULTS = []
ALL_SCORES = {}  # (dataset, algo) -> list of test MSEs over n_runs trials
HP_SEARCH_RESULTS = []
SELECTED_HPARAMS = {}
SELECTED_HP_ROWS = []

for dname in ACTIVE_DATASETS:
    print(f"\n>>> Dataset: {dname}")
    try:
        set_all_seeds(HP_SEARCH_SEED)
        res = get_loader(dname)
        if res is None: raise ValueError("None")
        (t1, v1, f1), (t2, v2, f2), dt = res
    except Exception as e: print(f"Load Fail: {e}"); continue

    # Final training uses full train+val; validation is used only for
    # the optional one-seed hyperparameter search above.
    full_l1 = DataLoader(f1, 256, True)
    full_l2 = DataLoader(f2, 256, True)
    full_all = DataLoader(torch.utils.data.ConcatDataset([f1, f2]), 256, True)
    test_loader = DataLoader(dt, 256, False)

    d_in = f1.x.shape[1]
    row = {"Dataset": dname}
    SELECTED_HPARAMS[dname] = {}

    if RUN_HYPERPARAM_SEARCH:
        print(f"  Hyperparameter search on validation split (seed={HP_SEARCH_SEED})")
        for algo in ACTIVE_ALGOS:
            if algo in ACTIVE_HP_SEARCH_ALGOS:
                best_params, best_val, search_rows = _run_hyperparameter_search(
                    dname, d_in, algo, t1, t2, v1, v2,
                )
                SELECTED_HPARAMS[dname][algo] = best_params
                HP_SEARCH_RESULTS.extend(search_rows)
                SELECTED_HP_ROWS.append({
                    "dataset": dname,
                    "algo": algo,
                    "selected_val_mse": best_val,
                    "params": dict(best_params),
                })
            else:
                SELECTED_HPARAMS[dname][algo] = dict(DEFAULT_HPARAMS[dname][algo])
                print(f"  [{algo}] using DEFAULT_HPARAMS: {SELECTED_HPARAMS[dname][algo]}")
    else:
        print("  Skipping hyperparameter search; using DEFAULT_HPARAMS")
        for algo in ACTIVE_ALGOS:
            SELECTED_HPARAMS[dname][algo] = dict(DEFAULT_HPARAMS[dname][algo])

    for algo in ACTIVE_ALGOS:
        p = SELECTED_HPARAMS[dname][algo]
        print(f"  [{algo}] final params={p}  (n_runs={n_runs})")
        trial_scores = []
        for seed in FINAL_SEEDS:
            set_all_seeds(seed)
            m_fin = get_model(dname, d_in).to(device)
            m_fin = train_loop(m_fin, full_all, (full_l1, full_l2),
                               algo, p, epochs=FINAL_EPOCHS)
            trial_scores.append(evaluate(m_fin, test_loader))
        ALL_SCORES[(dname, algo)] = trial_scores
        mean = float(np.mean(trial_scores))
        if n_runs > 1:
            std = float(np.std(trial_scores, ddof=1))
            row[algo] = f"{mean:.4f}±{std:.4f}"
            print(f"    {algo}: {mean:.4f} ± {std:.4f}  (over {n_runs} runs)")
        else:
            row[algo] = f"{mean:.4f}"
            print(f"    {algo}: {mean:.4f}")
    RESULTS.append(row)

if SELECTED_HP_ROWS:
    print("\n" + "="*60)
    print("SELECTED HYPERPARAMETERS")
    print("="*60)
    print(pd.DataFrame(SELECTED_HP_ROWS))

if HP_SEARCH_RESULTS:
    HP_SEARCH_DF = pd.DataFrame(HP_SEARCH_RESULTS)

print("\n" + "="*60)
print("FINAL RESULTS (MSE):")
print("="*60)
print(pd.DataFrame(RESULTS))
